In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(67)
num = np.linspace(0, 2 * np.pi, 100)
G = 1000
phase = rng.uniform(0, 2 * np.pi, G)
intercept = rng.uniform(1, 3, G)
genes = np.sin(num[:, None] + phase[None, :]) + intercept[None, :]

plt.scatter(num, genes[:, 0], color = 'blue', s=15)
plt.scatter(num, genes[:, 1], color = 'pink', s=15)
plt.show()

In [ ]:
lib = 5
noisy = rng.poisson(genes * lib) / lib
mask = rng.random(genes.shape) < 0.5
cells = np.where(mask, noisy, 0)

plt.scatter(num, cells[:, 0], color = 'blue', s=15)
plt.scatter(num, cells[:, 1], color = 'pink', s=15)
plt.show()

In [ ]:
from scipy.spatial.distance import cdist
from sklearn.decomposition import PCA

n_pca = 2
knn = 5
knn_max = 3 * knn
decay = 1

Z = PCA(n_pca).fit_transform(cells)
D = cdist(Z, Z)
order = np.argsort(D, axis=1)
K = np.zeros_like(D)
for i in range(len(cells)):
    sigma = D[i, order[i, knn - 1]]
    idx = order[i, :knn_max]
    K[i, idx] = np.exp(-(D[i, idx] / sigma) ** decay)
K = (K + K.T) / 2
K.shape

In [ ]:
M = K / K.sum(axis=1, keepdims=True)
M.sum(axis=1)[:5]

In [ ]:
t = 3
Mt = np.linalg.matrix_power(M, t)

cells_magic = Mt @ cells

observed = (cells != 0).astype(float)
filled = (Mt @ cells) / np.maximum(Mt @ observed, 1e-12)
cells_hat = np.where(cells == 0, filled, cells)
cells_hat.shape

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, X, title in [(axes[0], cells_magic, "MAGIC"), (axes[1], cells_hat, "zeros only")]:
    ax.scatter(num, X[:, 0], color = 'blue', s=15)
    ax.scatter(num, X[:, 1], color = 'pink', s=15)
    ax.plot(num, genes[:, 0], color = 'blue', alpha = 0.3)
    ax.plot(num, genes[:, 1], color = 'pink', alpha = 0.3)
    ax.set_title(title)
plt.show()